In [1]:
import pandas as pd
from phoenix.client import Client

client = Client()

spans_df = client.spans.get_spans_dataframe(
    project_identifier="shopbot-agent-0721"
)

print(f"Total spans: {len(spans_df)}")
display(spans_df.head())

e:\Unfold Data Science\LinkedIn\ai-observability-monitoring-and-debugging-llms-in-production-27494003\aiobserve\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Total spans: 122


,name,span_kind,parent_id,start_time,end_time,status_code,status_message,events,context.span_id,context.trace_id,...,attributes.openinference.span.kind,attributes.llm.token_count.completion,attributes.input.mime_type,attributes.llm.finish_reason,attributes.llm.token_count.completion_details.reasoning,attributes.llm.token_count.prompt_details.cache_read,attributes.llm.model_name,attributes.metadata,attributes.llm.tools,attributes.tool.name
context.span_id,,,,,,,,,,,,,,,,,,,,,
b90fe9f5372748a7,ChatCompletion,LLM,ad003954bb681964,2026-07-18 16:48:26.688371+00:00,2026-07-18 16:48:27.646086+00:00,OK,,[],b90fe9f5372748a7,58a4dce19b150679f261861c1294946d,...,LLM,48.0,application/json,stop,0.0,0.0,gpt-4o-mini-2024-07-18,None,None,NaN
c9cefa0ad2952161,document_relevance.evaluate,EVALUATOR,NaN,2026-07-18 16:48:26.687197+00:00,2026-07-18 16:48:27.651669+00:00,UNSET,,[],c9cefa0ad2952161,58a4dce19b150679f261861c1294946d,...,EVALUATOR,NaN,NaN,NaN,NaN,NaN,NaN,"{'evaluator.kind': 'llm', 'evaluator.class': '...",None,NaN
ad003954bb681964,LLM.generate_object,LLM,c9cefa0ad2952161,2026-07-18 16:48:26.687197+00:00,2026-07-18 16:48:27.649605+00:00,UNSET,,[],ad003954bb681964,58a4dce19b150679f261861c1294946d,...,LLM,NaN,NaN,NaN,NaN,NaN,gpt-4o-mini,None,None,NaN
e99097e70a1b0120,ChatCompletion,LLM,9b77b5663fa06609,2026-07-18 16:48:25.603645+00:00,2026-07-18 16:48:26.678296+00:00,OK,,[],e99097e70a1b0120,3928bf7a1885b82c913d5438a8009f24,...,LLM,68.0,application/json,stop,0.0,0.0,gpt-4o-mini-2024-07-18,None,None,NaN
9b77b5663fa06609,LLM.generate_object,LLM,53e519b499130f92,2026-07-18 16:48:25.602645+00:00,2026-07-18 16:48:26.681252+00:00,UNSET,,[],9b77b5663fa06609,3928bf7a1885b82c913d5438a8009f24,...,LLM,NaN,NaN,NaN,NaN,NaN,gpt-4o-mini,None,None,NaN


In [2]:
print("Available Span Types:")
display(spans_df["span_kind"].value_counts())

print("\nAvailable Columns:")
display(pd.Series(spans_df.columns))

Available Span Types:


span_kind
LLM          71
EVALUATOR    24
TOOL         15
AGENT        12
Name: count, dtype: int64


Available Columns:


0                                                  name
1                                             span_kind
2                                             parent_id
3                                            start_time
4                                              end_time
5                                           status_code
6                                        status_message
7                                                events
8                                       context.span_id
9                                      context.trace_id
10                              attributes.output.value
11                               attributes.input.value
12    attributes.llm.token_count.completion_details....
13                                attributes.llm.system
14      attributes.llm.token_count.prompt_details.audio
15                        attributes.llm.input_messages
16                       attributes.llm.output_messages
17                          attributes.output.mi

In [3]:
tool_spans = spans_df[
    spans_df["span_kind"] == "TOOL"
].copy()

tool_spans["tool_name"] = tool_spans["name"].str.replace(
    "tool.", "", regex=False
)

display(
    tool_spans[
        [
            "context.trace_id",
            "tool_name",
            "attributes.output.value"
        ]
    ].head()
)

,context.trace_id,tool_name,attributes.output.value
context.span_id,,,
cf2e67e0b05ab9b2,f38f670b07c6d274be52e9a1564daed8,check_return_eligibility,"{""eligible"": true, ""days_since_delivery"": 5, ""..."
98dd0c9c47c02cdf,f38f670b07c6d274be52e9a1564daed8,check_order_status,"{""status"": ""delivered"", ""delivered_date"": ""202..."
c520155133115e36,94e4502ae428988d3425b9c0bc8a754e,process_refund,"{""status"": ""refund_initiated"", ""order_id"": ""OR..."
56e4fb82b27bb78a,94e4502ae428988d3425b9c0bc8a754e,check_return_eligibility,"{""eligible"": true, ""days_since_delivery"": 5, ""..."
3a6411513e15ed85,94e4502ae428988d3425b9c0bc8a754e,check_order_status,"{""status"": ""delivered"", ""delivered_date"": ""202..."


In [4]:
eval_results = pd.read_csv("shopbot_eval_results.csv")

print(eval_results.shape)

display(eval_results.head())

(12, 10)


,context.trace_id,context.span_id,input,output,faithfulness_label,faithfulness_value,faithfulness_explanation,relevance_label,relevance_value,relevance_explanation
0,de6756ca67006a78e865cc0c90257fd5,9da371298fd291bd,"{""messages"": [{""role"": ""system"", ""content"": ""Y...","{""id"":""chatcmpl-E32Trm55y3PKMP30U3u4BUlCyQIct""...",faithful,1.0,The response accurately summarizes the return ...,unrelated,0.0,The document text is empty and does not provid...
1,b51f1142a9b510bf7e72cf11fa8886ed,50afcd975388273c,"{""messages"": [{""role"": ""system"", ""content"": ""Y...","{""id"":""chatcmpl-E32Tt15F9gM1YjQtkCnS7uLv0IeHu""...",faithful,1.0,The response accurately requests the order ID ...,unrelated,0.0,"The document text is empty, therefore it does ..."
2,38fdcd67906cc646212608dcbaf65b7e,2f7a31d395c39cd2,"{""messages"": [{""role"": ""system"", ""content"": ""Y...","{""id"":""chatcmpl-E32TwXTauO8r93EsFObPSIK0e74HM""...",faithful,1.0,The response correctly states that StyleHive d...,relevant,1.0,The document text contains information about t...
3,7ed407adee06e9d28bca8cfcf998a0b0,e21d07c3471d444d,"{""messages"": [{""role"": ""system"", ""content"": ""Y...","{""id"":""chatcmpl-E32TywYCTuI8vVGTmMEZ092TlnkHM""...",faithful,1.0,The response correctly states that order ORD-1...,relevant,1.0,The document text states that the order is not...
4,bfddd32ddf8b050c2ffb64b14ae82218,7f4f5142d54f1744,"{""messages"": [{""role"": ""system"", ""content"": ""Y...","{""id"":""chatcmpl-E32U1z831CbODnI9jb8RhI2dPdhV4""...",faithful,1.0,The response accurately states that the return...,relevant,1.0,The document text provides specific informatio...


In [6]:
last_tool = (
    tool_spans
    .sort_values("start_time")
    .groupby("context.trace_id")
    .tail(1)
    [
        [
            "context.trace_id",
            "tool_name"
        ]
    ]
)

analysis_df = eval_results.merge(
    last_tool,
    left_on="context.trace_id",
    right_on="context.trace_id",
    how="left"
)

display(
    analysis_df[
        [
            "tool_name",
            "faithfulness_label",
            "relevance_label"
        ]
    ].head()
)

,tool_name,faithfulness_label,relevance_label
0,NaN,faithful,unrelated
1,NaN,faithful,unrelated
2,check_order_status,faithful,relevant
3,check_return_eligibility,faithful,relevant
4,check_return_eligibility,faithful,relevant


In [7]:
tool_summary = (
    analysis_df
    .groupby("tool_name")
    .agg(
        Total=("tool_name","count"),
        Faithfulness_Fails=(
            "faithfulness_label",
            lambda x: (x=="unfaithful").sum()
        ),
        Relevance_Fails=(
            "relevance_label",
            lambda x: (x=="unrelated").sum()
        )
    )
    .sort_values(
        "Faithfulness_Fails",
        ascending=False
    )
)

tool_summary

,Total,Faithfulness_Fails,Relevance_Fails
tool_name,,,
check_order_status,3,1,0
check_return_eligibility,4,0,0
process_refund,2,0,0


In [8]:
refund_failures = analysis_df[
    (analysis_df["tool_name"]=="process_refund") &
    (analysis_df["faithfulness_label"]=="unfaithful")
]

refund_failures[
    [
        "context.trace_id",
        "input",
        "output",
        "faithfulness_explanation"
    ]
]

,context.trace_id,input,output,faithfulness_explanation


In [9]:
relevance_failures = analysis_df[
    analysis_df["relevance_label"]=="unrelated"
]

relevance_failures[
    [
        "tool_name",
        "input",
        "output",
        "relevance_explanation"
    ]
]

,tool_name,input,output,relevance_explanation
0,NaN,"{""messages"": [{""role"": ""system"", ""content"": ""Y...","{""id"":""chatcmpl-E32Trm55y3PKMP30U3u4BUlCyQIct""...",The document text is empty and does not provid...
1,NaN,"{""messages"": [{""role"": ""system"", ""content"": ""Y...","{""id"":""chatcmpl-E32Tt15F9gM1YjQtkCnS7uLv0IeHu""...","The document text is empty, therefore it does ..."
10,NaN,"{""messages"": [{""role"": ""system"", ""content"": ""Y...","{""id"":""chatcmpl-E32UJPVYjoKy6ERz6ScNVib0YTeP2""...","The document text is empty, so it does not con..."


In [10]:
summary = pd.DataFrame({
    "Metric":[
        "Total Conversations",
        "Faithfulness Fails",
        "Relevance Fails"
    ],
    "Count":[
        len(analysis_df),
        (analysis_df["faithfulness_label"]=="unfaithful").sum(),
        (analysis_df["relevance_label"]=="unrelated").sum()
    ]
})

summary

,Metric,Count
0,Total Conversations,12
1,Faithfulness Fails,1
2,Relevance Fails,3


In [11]:
analysis_df.groupby(
    [
        "tool_name",
        "faithfulness_label",
        "relevance_label"
    ]
).size().reset_index(name="Count")

,tool_name,faithfulness_label,relevance_label,Count
0,check_order_status,faithful,relevant,2
1,check_order_status,unfaithful,relevant,1
2,check_return_eligibility,faithful,relevant,4
3,process_refund,faithful,relevant,2


In [12]:
analysis_df

,context.trace_id,context.span_id,input,output,faithfulness_label,faithfulness_value,faithfulness_explanation,relevance_label,relevance_value,relevance_explanation,tool_name
0,de6756ca67006a78e865cc0c90257fd5,9da371298fd291bd,"{""messages"": [{""role"": ""system"", ""content"": ""Y...","{""id"":""chatcmpl-E32Trm55y3PKMP30U3u4BUlCyQIct""...",faithful,1.0,The response accurately summarizes the return ...,unrelated,0.0,The document text is empty and does not provid...,NaN
1,b51f1142a9b510bf7e72cf11fa8886ed,50afcd975388273c,"{""messages"": [{""role"": ""system"", ""content"": ""Y...","{""id"":""chatcmpl-E32Tt15F9gM1YjQtkCnS7uLv0IeHu""...",faithful,1.0,The response accurately requests the order ID ...,unrelated,0.0,"The document text is empty, therefore it does ...",NaN
2,38fdcd67906cc646212608dcbaf65b7e,2f7a31d395c39cd2,"{""messages"": [{""role"": ""system"", ""content"": ""Y...","{""id"":""chatcmpl-E32TwXTauO8r93EsFObPSIK0e74HM""...",faithful,1.0,The response correctly states that StyleHive d...,relevant,1.0,The document text contains information about t...,check_order_status
3,7ed407adee06e9d28bca8cfcf998a0b0,e21d07c3471d444d,"{""messages"": [{""role"": ""system"", ""content"": ""Y...","{""id"":""chatcmpl-E32TywYCTuI8vVGTmMEZ092TlnkHM""...",faithful,1.0,The response correctly states that order ORD-1...,relevant,1.0,The document text states that the order is not...,check_return_eligibility
4,bfddd32ddf8b050c2ffb64b14ae82218,7f4f5142d54f1744,"{""messages"": [{""role"": ""system"", ""content"": ""Y...","{""id"":""chatcmpl-E32U1z831CbODnI9jb8RhI2dPdhV4""...",faithful,1.0,The response accurately states that the return...,relevant,1.0,The document text provides specific informatio...,check_return_eligibility
5,0b6277f12bd8accdac7a2b1cf0133900,e24eba753120c6ca,"{""messages"": [{""role"": ""system"", ""content"": ""Y...","{""id"":""chatcmpl-E32U35BtLQqyzDP4hx0kFFOm0Mv8o""...",faithful,1.0,The response correctly states that Order ORD-1...,relevant,1.0,The document text states that the order is not...,check_return_eligibility
6,2cffeee19ce04e76cdb440057c5444f4,1163f40431c6e577,"{""messages"": [{""role"": ""system"", ""content"": ""Y...","{""id"":""chatcmpl-E32U6ImN3kRIA4zlG7gyaoOEqk3VL""...",unfaithful,0.0,The response incorrectly states that ORD-1001 ...,relevant,1.0,The document contains the statuses of two spec...,check_order_status
7,b066acea2c23235deeedb57a58b195b8,352bf6356e7754bc,"{""messages"": [{""role"": ""system"", ""content"": ""Y...","{""id"":""chatcmpl-E32U916MJRcoTkcF741GH1aWf6Wn6""...",faithful,1.0,The assistant correctly states that there is n...,relevant,1.0,The document text indicates that there is no o...,check_order_status
8,3b50630500f4c9e9c319f5f177c38c3a,3a76a6d3907bfbde,"{""messages"": [{""role"": ""system"", ""content"": ""Y...","{""id"":""chatcmpl-E32UEKc6sM1Mrgrl8mHG66Nnkit2g""...",faithful,1.0,The response correctly confirms that the refun...,relevant,1.0,The document text confirms that the refund for...,process_refund
9,94e4502ae428988d3425b9c0bc8a754e,105009113bcc3777,"{""messages"": [{""role"": ""system"", ""content"": ""Y...","{""id"":""chatcmpl-E32UIX7aYmYstdzMa3fnXMGUWNBHj""...",faithful,1.0,The response correctly states that order ORD-1...,relevant,1.0,The document text provides the status of the r...,process_refund
